In [ ]:
# | default_exp seo_site_analysis

# SEO Site Analysis
> Crawl and analyze entire sites for technical SEO issues and link structure.

In [ ]:
# | export
from sqlmodel import Session, select
from seo_rat.article import Article,get_articles_by_website
from seo_rat.content_parser import remove_metadata, normalize_text, calculate_similarity
from seo_rat.seo_content_analysis import calculate_keyword_density

In [ ]:
# | export
def detect_duplicate_content(session:Session,              # Active database session
                              website_id:int,               # Website to search within
                              content:str,                  # Normalized page content
                              file_path:str,                # Current article path (to exclude self)
                              similarity_threshold:float=0.8 # Minimum similarity to flag
                             ) -> dict:
    "Find articles within a website with content similar to the given content."
    current = normalize_text(remove_metadata(content))
    similar = []
    for article in get_articles_by_website(session, website_id):
        if article.file_path == file_path: continue
        other = normalize_text(remove_metadata(open(article.file_path).read()))
        if (sim := calculate_similarity(current, other)) >= similarity_threshold:
            similar.append({"file_path": article.file_path, "similarity": sim})
    return {"has_duplicates": len(similar) > 0, "similar_articles": similar}


In [ ]:
# | export
def analyze_keyword_cannibalization(session:Session, # Active database session
                                    website_id:int,  # Website to search within
                                    keyword:str      # Focus keyword to check
                                   ) -> dict:
    "Find articles within a website competing for the same focus keyword."
    articles = [a for a in get_articles_by_website(session, website_id)
                if a.focus_keyword == keyword]
    if len(articles) <= 1:
        return {"has_cannibalization": False, "keyword": keyword, "count": len(articles)}
    results = [{"file_path": a.file_path,
                **{k: v for k, v in calculate_keyword_density(
                    remove_metadata(open(a.file_path).read()), keyword).items()
                   if k in ("density", "count")}}
               for a in articles]
    return {"has_cannibalization": True, "keyword": keyword,
            "count": len(articles), "articles": results}


def analyze_content_groups(session:Session,              # Active database session
                           website_id:int,               # Website to search within
                           similarity_threshold:float=0.8 # Minimum similarity to group
                          ) -> dict:
    "Group similar articles together across a website."
    articles = get_articles_by_website(session, website_id)
    groups, processed = [], set()
    for article in articles:
        if article.id in processed: continue
        main = normalize_text(remove_metadata(open(article.file_path).read()))
        group = {"main_article": article.file_path, "similar_articles": []}
        for other in articles:
            if other.id == article.id or other.id in processed: continue
            other_content = normalize_text(remove_metadata(open(other.file_path).read()))
            if (sim := calculate_similarity(main, other_content)) >= similarity_threshold:
                group["similar_articles"].append({"file_path": other.file_path, "similarity": sim})
                processed.add(other.id)
        if group["similar_articles"]:
            groups.append(group)
            processed.add(article.id)
    return {"total_articles": len(articles), "groups": groups, "duplicate_groups": len(groups)}


In [ ]:
#| export
import hashlib

def get_shingles(text: str, n: int = 3) -> set:
    "Generate n-word shingles from text."
    words = text.lower().split()
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def minhash_signature(shingles: set, num_hashes: int = 128) -> list[int]:
    "Compute MinHash signature for a set of shingles."
    signature = []
    for seed in range(num_hashes):
        min_hash = min(
            int(hashlib.md5(f"{seed}:{s}".encode()).hexdigest(), 16)
            for s in shingles
        ) if shingles else 0
        signature.append(min_hash)
    return signature

def minhash_similarity(sig1: list[int], sig2: list[int]) -> float:
    "Estimate Jaccard similarity from two MinHash signatures."
    return sum(a == b for a, b in zip(sig1, sig2)) / len(sig1)

def analyze_content_groups_fast(session: Session, website_id: int,
                                 similarity_threshold: float = 0.8,
                                 num_hashes: int = 128) -> dict:
    "Group similar articles using MinHash for fast approximate similarity."
    articles = get_articles_by_website(session, website_id)
    sigs = {}
    for a in articles:
        try:
            text = remove_metadata(open(a.file_path).read())
            sigs[a.id] = (a.file_path, minhash_signature(get_shingles(text), num_hashes))
        except Exception:
            continue

    groups, processed = [], set()
    ids = list(sigs.keys())
    for i, id1 in enumerate(ids):
        if id1 in processed: continue
        path1, sig1 = sigs[id1]
        group = {"main_article": path1, "similar_articles": []}
        for id2 in ids[i+1:]:
            if id2 in processed: continue
            path2, sig2 = sigs[id2]
            sim = minhash_similarity(sig1, sig2)
            if sim >= similarity_threshold:
                group["similar_articles"].append({"file_path": path2, "similarity": sim})
                processed.add(id2)
        if group["similar_articles"]:
            groups.append(group)
            processed.add(id1)

    return {"total_articles": len(articles), "groups": groups, "duplicate_groups": len(groups)}


NameError: name 'Session' is not defined

In [ ]:
# | hide
#| eval: false
from fastcore.test import test_eq
from pprint import pprint
from sqlmodel import create_engine, Session, SQLModel
from seo_rat.models import Website
from seo_rat.article import Article, insert_article
import tempfile, os
from pathlib import Path


In [ ]:
# | hide
#| eval: false
sample_dir = Path("sample")
if not sample_dir.exists():
    sample_dir = Path("../sample")

content = (sample_dir / "example.md").read_text()

engine = create_engine("sqlite:///:memory:")
SQLModel.metadata.create_all(engine)

with Session(engine) as session:
    website = Website(url="https://test.com", name="Test", lang="en")
    session.add(website)
    session.commit()
    session.refresh(website)

    with tempfile.NamedTemporaryFile(mode="w", suffix=".md", delete=False) as f1:
        f1.write(content)
        path1 = f1.name

    with tempfile.NamedTemporaryFile(mode="w", suffix=".md", delete=False) as f2:
        f2.write(content)
        path2 = f2.name

    insert_article(session, Article(website_id=website.id, file_path=path1, focus_keyword="Kareem"))
    insert_article(session, Article(website_id=website.id, file_path=path2, focus_keyword="Kareem"))

    # detect_duplicate_content
    dup_result = detect_duplicate_content(session, website.id, content, path1)
    test_eq(dup_result["has_duplicates"], True)
    pprint(dup_result)

    # analyze_keyword_cannibalization
    cannibal_result = analyze_keyword_cannibalization(session, website.id, "Kareem")
    test_eq(cannibal_result["has_cannibalization"], True)
    test_eq(cannibal_result["count"], 2)
    pprint(cannibal_result)

    # analyze_content_groups
    groups_result = analyze_content_groups(session, website.id, 0.8)
    test_eq(groups_result["duplicate_groups"] >= 1, True)
    pprint(groups_result)

    os.unlink(path1)
    os.unlink(path2)


{'has_duplicates': True,
 'similar_articles': [{'file_path': '/tmp/tmp6tohuqfm.md', 'similarity': 1.0}]}
{'articles': [{'count': 3,
               'density': 2.4193548387096775,
               'file_path': '/tmp/tmp9js08as0.md'},
              {'count': 3,
               'density': 2.4193548387096775,
               'file_path': '/tmp/tmp6tohuqfm.md'}],
 'count': 2,
 'has_cannibalization': True,
 'keyword': 'Kareem'}
{'duplicate_groups': 1,
 'groups': [{'main_article': '/tmp/tmp9js08as0.md',
             'similar_articles': [{'file_path': '/tmp/tmp6tohuqfm.md',
                                   'similarity': 1.0}]}],
 'total_articles': 2}
